# 01 — APOGEE DR17 Data Access

Verify access to the APOGEE DR17 allStarLite catalog, explore the schema, and demonstrate loading individual stellar spectra.

**Data source:** SDSS-IV APOGEE-2, Data Release 17
- allStarLite catalog: ~650K stars with stellar parameters, RVs, and visit metadata
- apStar files: multi-visit H-band spectra (1.51–1.70 μm, R~22,500, 8575 pixels)

In [ ]:
import os
import sys
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
from astropy.table import Table

# Add project root to path
sys.path.insert(0, os.path.abspath("../.."))
from config import DATA_CONFIG, LABEL_CONFIG
from src.data import download_allstar_catalog, get_visit_spectra

%matplotlib inline
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["figure.dpi"] = 120

## 1. Download and Load the allStarLite Catalog

The allStarLite catalog is a compact version of the full allStar file, containing stellar parameters (Teff, logg, [Fe/H]), radial velocities, visit counts, S/N, and quality flags for every APOGEE star.

In [ ]:
# Download or load from cache/Fornax shared storage
catalog = download_allstar_catalog()
print(f"Catalog shape: {len(catalog)} stars, {len(catalog.colnames)} columns")

In [ ]:
# Explore schema — key columns for our project
key_columns = [
    "APOGEE_ID", "RA", "DEC", "TELESCOPE",
    "TEFF", "LOGG", "FE_H",           # stellar parameters
    "VHELIO_AVG", "VSCATTER",          # radial velocity
    "NVISITS", "SNR",                  # observation metadata
    "STARFLAG", "ASPCAPFLAG",          # quality flags
    "FIELD",                           # field name (needed for apStar download)
]

print("Key columns and types:")
for col in key_columns:
    if col in catalog.colnames:
        print(f"  {col:20s}  dtype={catalog[col].dtype}  example={catalog[col][0]}")
    else:
        print(f"  {col:20s}  ** NOT FOUND **")

## 2. Basic Statistics

In [ ]:
print(f"Total stars: {len(catalog):,}")
print(f"\nNVISITS distribution:")
for threshold in [1, 2, 3, 5, 10, 20, 50]:
    count = (catalog["NVISITS"] >= threshold).sum()
    print(f"  >= {threshold:3d} visits: {count:>8,} stars ({100*count/len(catalog):.1f}%)")

print(f"\nSNR distribution:")
for threshold in [10, 25, 50, 100, 200]:
    count = (catalog["SNR"] >= threshold).sum()
    print(f"  >= {threshold:3d}: {count:>8,} stars ({100*count/len(catalog):.1f}%)")

print(f"\nTeff range: {np.nanmin(catalog['TEFF']):.0f} — {np.nanmax(catalog['TEFF']):.0f} K")
print(f"logg range: {np.nanmin(catalog['LOGG']):.2f} — {np.nanmax(catalog['LOGG']):.2f}")
print(f"VSCATTER range: {np.nanmin(catalog['VSCATTER']):.3f} — {np.nanmax(catalog['VSCATTER']):.1f} km/s")

## 3. Kiel Diagram (Teff vs. logg)

The Kiel diagram is the spectroscopic equivalent of the HR diagram. APOGEE is dominated by red giants (low logg, cool Teff) due to its infrared selection, but also includes dwarfs and subgiants.

In [ ]:
# Kiel diagram colored by metallicity
good = (catalog["TEFF"] > 0) & (catalog["LOGG"] > -1) & np.isfinite(catalog["FE_H"])

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Left: colored by [Fe/H]
sc = axes[0].scatter(
    catalog["TEFF"][good], catalog["LOGG"][good],
    c=catalog["FE_H"][good], cmap="RdYlBu_r", s=0.1, alpha=0.3,
    vmin=-2, vmax=0.5
)
axes[0].invert_xaxis()
axes[0].invert_yaxis()
axes[0].set_xlabel("Teff (K)")
axes[0].set_ylabel("log g (dex)")
axes[0].set_title("Kiel Diagram — colored by [Fe/H]")
plt.colorbar(sc, ax=axes[0], label="[Fe/H]")

# Right: colored by VSCATTER (our binary indicator)
has_vs = good & (catalog["VSCATTER"] > 0) & (catalog["NVISITS"] >= 3)
sc2 = axes[1].scatter(
    catalog["TEFF"][has_vs], catalog["LOGG"][has_vs],
    c=np.log10(catalog["VSCATTER"][has_vs]), cmap="plasma", s=0.3, alpha=0.3,
    vmin=-1, vmax=2
)
axes[1].invert_xaxis()
axes[1].invert_yaxis()
axes[1].set_xlabel("Teff (K)")
axes[1].set_ylabel("log g (dex)")
axes[1].set_title("Kiel Diagram — colored by log10(VSCATTER)")
plt.colorbar(sc2, ax=axes[1], label="log10(VSCATTER) [km/s]")

plt.tight_layout()
plt.savefig("../../figures/kiel_diagram.png", bbox_inches="tight")
plt.show()

## 4. NVISITS and VSCATTER Distributions

These are the two most important columns for our project:
- **NVISITS**: how many times APOGEE observed each star (need >=3 for label generation)
- **VSCATTER**: the standard deviation of per-visit RVs (our primary binary indicator)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# NVISITS histogram
axes[0].hist(catalog["NVISITS"], bins=np.arange(0.5, 60.5, 1), edgecolor="black", linewidth=0.3)
axes[0].set_xlabel("NVISITS")
axes[0].set_ylabel("Number of Stars")
axes[0].set_title("Distribution of Visit Counts")
axes[0].set_yscale("log")
axes[0].axvline(3, color="red", linestyle="--", label=f"min_visits={LABEL_CONFIG['min_visits']}")
axes[0].legend()

# VSCATTER histogram (for multi-visit stars)
multi = catalog["NVISITS"] >= 3
vs = catalog["VSCATTER"][multi]
vs = vs[vs > 0]
axes[1].hist(np.log10(vs), bins=100, edgecolor="black", linewidth=0.3)
axes[1].set_xlabel("log10(VSCATTER) [km/s]")
axes[1].set_ylabel("Number of Stars")
axes[1].set_title(f"RV Scatter Distribution (NVISITS >= 3, N={multi.sum():,})")
axes[1].axvline(np.log10(LABEL_CONFIG["rv_scatter_single_threshold"]), color="green",
                linestyle="--", label=f"Single threshold ({LABEL_CONFIG['rv_scatter_single_threshold']} km/s)")
axes[1].axvline(np.log10(LABEL_CONFIG["rv_scatter_binary_threshold"]), color="red",
                linestyle="--", label=f"Binary threshold ({LABEL_CONFIG['rv_scatter_binary_threshold']} km/s)")
axes[1].legend()

plt.tight_layout()
plt.savefig("../../figures/nvisits_vscatter_dist.png", bbox_inches="tight")
plt.show()

## 5. Load an Example apStar Spectrum

Demonstrate loading multi-visit spectra for a single star. The apStar file contains a combined (coadded) spectrum plus individual visit spectra.

In [ ]:
# Pick a well-observed star with high S/N and many visits
multi_visit = catalog[(catalog["NVISITS"] >= 10) & (catalog["SNR"] >= 200)]
print(f"Stars with >= 10 visits and SNR >= 200: {len(multi_visit)}")

# Sort by VSCATTER to pick one high-VSCATTER (likely binary) and one low (likely single)
multi_visit.sort("VSCATTER")

# Example single star (low VSCATTER)
single_star = multi_visit[0]
print(f"\nExample single star:")
print(f"  APOGEE_ID: {single_star['APOGEE_ID']}")
print(f"  NVISITS: {single_star['NVISITS']}, SNR: {single_star['SNR']:.0f}")
print(f"  VSCATTER: {single_star['VSCATTER']:.3f} km/s")
print(f"  Teff: {single_star['TEFF']:.0f} K, logg: {single_star['LOGG']:.2f}")

# Example binary candidate (high VSCATTER)
binary_star = multi_visit[-1]
print(f"\nExample binary candidate:")
print(f"  APOGEE_ID: {binary_star['APOGEE_ID']}")
print(f"  NVISITS: {binary_star['NVISITS']}, SNR: {binary_star['SNR']:.0f}")
print(f"  VSCATTER: {binary_star['VSCATTER']:.3f} km/s")
print(f"  Teff: {binary_star['TEFF']:.0f} K, logg: {binary_star['LOGG']:.2f}")

In [ ]:
# Load the apStar file for the single star
# Note: on first run this will download the file (~2 MB)
try:
    field = single_star["FIELD"] if "FIELD" in single_star.colnames else None
    telescope = single_star["TELESCOPE"] if "TELESCOPE" in single_star.colnames else "apo25m"
    
    spectra = get_visit_spectra(
        single_star["APOGEE_ID"].strip(),
        telescope=telescope.strip() if isinstance(telescope, str) else "apo25m",
        field=field.strip() if isinstance(field, str) else None,
    )
    
    print(f"Loaded {spectra['n_visits']} visits")
    print(f"Wavelength range: {spectra['wavelength'][0]:.1f} — {spectra['wavelength'][-1]:.1f} Angstrom")
    print(f"Flux shape: {spectra['flux'].shape}")
    print(f"Per-visit RVs: {spectra['rv_per_visit']}")
    
    # Plot combined + individual visit spectra
    fig, axes = plt.subplots(2, 1, figsize=(14, 8))
    
    # Combined spectrum
    axes[0].plot(spectra["wavelength"], spectra["flux_combined"], "k-", linewidth=0.5)
    axes[0].set_ylabel("Flux")
    axes[0].set_title(f"Combined Spectrum — {single_star['APOGEE_ID']} (Single, VSCATTER={single_star['VSCATTER']:.3f} km/s)")
    axes[0].set_xlim(15150, 16950)
    
    # Individual visits (first 5)
    n_show = min(5, spectra["n_visits"])
    for i in range(n_show):
        label = f"Visit {i+1} (RV={spectra['rv_per_visit'][i]:.2f} km/s)"
        axes[1].plot(spectra["wavelength"], spectra["flux"][i], linewidth=0.4, alpha=0.7, label=label)
    axes[1].set_xlabel("Wavelength (Angstrom)")
    axes[1].set_ylabel("Flux")
    axes[1].set_title("Individual Visit Spectra")
    axes[1].set_xlim(15150, 16950)
    axes[1].legend(fontsize=8)
    
    plt.tight_layout()
    plt.savefig("../../figures/example_single_star_spectra.png", bbox_inches="tight")
    plt.show()
    
except Exception as e:
    print(f"Could not load apStar file (expected if running locally without data): {e}")
    print("This will work on Fornax with access to the SDSS SAS.")

## 6. Fornax Access Notes

Document what worked/didn't for data access on this platform. Update this cell after running on Fornax.

- [ ] allStarLite via shared mount (`/data/sdss/...`)?
- [ ] allStarLite via HTTP download?
- [ ] apStar files via HTTP?
- [ ] astroquery SDSS module?
- [ ] Approximate download speeds?